In [0]:
state_under_test = "prepareForHearing"

from datetime import datetime, timedelta
from pyspark.sql.functions import input_file_name, regexp_extract, regexp_replace, col
from pyspark.sql import Row
import os
import pandas as pd
import ast
from dataclasses import dataclass
import pandas as pd

from pyspark.sql import functions as F

In [0]:
#######################
#Spark Config
#######################
config_path = "dbfs:/configs/config.json"
config = spark.read.option("multiline", "true").json(config_path)
first_row = config.first()
env = first_row["env"].strip().lower()
lz_key = first_row["lz_key"].strip().lower()
keyvault_name = f"ingest{lz_key}-meta002-{env}"
client_secret = dbutils.secrets.get(scope=keyvault_name, key='SERVICE-PRINCIPLE-CLIENT-SECRET')
tenant_id = dbutils.secrets.get(scope=keyvault_name, key='SERVICE-PRINCIPLE-TENANT-ID')
client_id = dbutils.secrets.get(scope=keyvault_name, key='SERVICE-PRINCIPLE-CLIENT-ID') 


config = spark.read.option("multiline", "true").json("dbfs:/configs/config.json")
env_name = config.first()["env"].strip().lower()
lz_key = config.first()["lz_key"].strip().lower()
 
print(f"env_code: {lz_key}")  # This won't be redacted
print(f"env_name: {env_name}")  # This won't be redacted
 
KeyVault_name = f"ingest{lz_key}-meta002-{env_name}"
print(f"KeyVault_name: {KeyVault_name}")
 
# Service principal credentials
client_id = dbutils.secrets.get(KeyVault_name, "SERVICE-PRINCIPLE-CLIENT-ID")
client_secret = dbutils.secrets.get(KeyVault_name, "SERVICE-PRINCIPLE-CLIENT-SECRET")
tenant_id = dbutils.secrets.get(KeyVault_name, "SERVICE-PRINCIPLE-TENANT-ID")
 
# Storage account names
curated_storage = f"ingest{lz_key}curated{env_name}"
checkpoint_storage = f"ingest{lz_key}xcutting{env_name}"
raw_storage = f"ingest{lz_key}raw{env_name}"
landing_storage = f"ingest{lz_key}landing{env_name}"
external_storage = f"ingest{lz_key}external{env_name}"
 
 
# Spark config for curated storage (Delta table)
spark.conf.set(f"fs.azure.account.auth.type.{curated_storage}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{curated_storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{curated_storage}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{curated_storage}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{curated_storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")
 
# Spark config for checkpoint storage
spark.conf.set(f"fs.azure.account.auth.type.{checkpoint_storage}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{checkpoint_storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{checkpoint_storage}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{checkpoint_storage}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{checkpoint_storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")
 
# Spark config for checkpoint storage
spark.conf.set(f"fs.azure.account.auth.type.{raw_storage}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{raw_storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{raw_storage}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{raw_storage}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{raw_storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")
 
# Spark config for checkpoint storage
spark.conf.set(f"fs.azure.account.auth.type.{landing_storage}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{landing_storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{landing_storage}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{landing_storage}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{landing_storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

 
# Spark config for checkpoint storage
spark.conf.set(f"fs.azure.account.auth.type.{external_storage}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{external_storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{external_storage}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{external_storage}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{external_storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")
 
# Setting variables for use in subsequent cells
bronze_path = f"abfss://bronze@ingest{lz_key}curated{env_name}.dfs.core.windows.net/ARIADM/ACTIVE/CCD/APPEALS/"
silver_path = f"abfss://silver@ingest{lz_key}curated{env_name}.dfs.core.windows.net/ARIADM/ACTIVE/CCD/APPEALS/"
audit_path = f"abfss://silver@ingest{lz_key}curated{env_name}.dfs.core.windows.net/ARIADM/ACTIVE/CCD/APPEALS/AUDIT/{state_under_test}"
gold_path = f"abfss://gold@ingest{lz_key}curated{env_name}.dfs.core.windows.net/ARIADM/ACTIVE/CCD/APPEALS/{state_under_test}"
 
# Print all variables
variables = {
    # "read_hive": read_hive,
    
    "bronze_path": bronze_path,
    "silver_path": silver_path,
    "audit_path": audit_path,
    "gold_path": gold_path,
    "key_vault": KeyVault_name,
    "state_under_test": state_under_test
 
}
 
display(variables)

In [0]:
data_path = f"abfss://silver@ingest{lz_key}curated{env}.dfs.core.windows.net/ARIADM/ACTIVE/CCD/AUDIT/APPEALS/all_active_states/ack_audit"

In [0]:
print(data_path)
df = spark.read.format("delta").load(data_path).filter(col("State") == state_under_test)
df.display()

In [0]:
display(df.count())

In [0]:
display(
df.groupBy("State", "AriaCaseNo" ).count().filter(col("count") > 1)
)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Define which case we are looking at and order by the newest time
# We use AriaCaseNo as the unique ID
window_spec = Window.partitionBy("AriaCaseNo").orderBy(F.desc("StartDateTime"))

# 2. Apply the dedupe logic
df_deduped = df.withColumn("row_number", F.row_number().over(window_spec)) \
               .filter(F.col("row_number") == 1) \
               .drop("row_number")

# 3. Verify it worked
print(f"Original Count: {df.count()}")
print(f"Deduplicated Count: {df_deduped.count()}")

# This should now show 0 rows (because all counts are 1)
display(
    df_deduped.groupBy("State", "AriaCaseNo").count().filter(F.col("count") > 1)
)

df_deduped.display()

In [0]:
import json

# UDF to grab the inner 'case_data' block
@F.udf(returnType=F.StringType())
def extract_inner_json(resp):
    if not resp: return None
    try:
        return json.dumps(json.loads(resp).get("case_data", {}))
    except: return None

# 1. Create a column with just the inner JSON string
audit_with_inner = df_deduped.withColumn("inner_json_str", extract_inner_json(F.col("SuccessResponse")))

# 2. Get a sample to build the schema automatically
sample_row = audit_with_inner.filter(F.col("inner_json_str").isNotNull()).select("inner_json_str").first()
if sample_row:
    auto_schema = F.schema_of_json(sample_row[0])
    
    # 3. Parse that string into a proper Struct (Columns!)
    audit_payload_df = audit_with_inner.withColumn("payload", F.from_json(F.col("inner_json_str"), auto_schema)) \
                                       .select("AriaCaseNo", "payload.*") # This expands everything!
else:
    print("No valid JSON found in SuccessResponse to parse")

In [0]:
audit_payload_df.display()

In [0]:
audit_payload_df.printSchema()

In [0]:
from pyspark.sql import functions as F

########################
# 1. Identify the Latest Gold Path
########################
json_location = dbutils.fs.ls(f"{gold_path}/")[-1]
latest_json_name = json_location.name
latest_json_path = f"{gold_path}/{latest_json_name}JSON/"

# Most CCD/Gold structures have a 'JSON' subfolder
gold_data_glob = f"{latest_json_path}JSON/*.json"
print(f"Loading Gold Data from: {gold_data_glob}")

########################
# 2. Load the JSON Data
########################
# multiLine=True is required for pretty-printed JSON files
raw_df = spark.read.option("multiLine", "true").json([latest_json_path]).withColumn("raw_filename", input_file_name())

########################
# 3. Clean & Deduplicate
########################
# We create 'clean_gold_ref' to act as the anchor for our join
latest_json_df = raw_df.withColumn(
    "clean_gold_ref", 
    F.upper(F.trim(F.col("appealReferenceNumber")))
).filter(F.col("clean_gold_ref").isNotNull())

# Just in case there are duplicate files for the same case in the Gold folder
latest_json_df = latest_json_df.dropDuplicates(["clean_gold_ref"])

print(f"Success - latest_json_df defined with {latest_json_df.count()} records.")

In [0]:
latest_json_df.printSchema()

In [0]:
# --- Step A: Expanded Audit Selection ---
audit_final = audit_payload_df.select(
    # KEY
    F.col("AriaCaseNo").alias("appealReferenceNumber"),
    
    # 1. ROOT STRINGS (The "Extended" Simple List)
    "appellantFullName", "appellantFamilyName", "appellantGivenNames",
    "appellantDateOfBirth", "appellantInUk", "appellantInDetention", "appellantStateless",
    "appealType", "appealTypeDescription", "appealOutOfCountry", "appealSubmissionDate",
    "feeAmountGbp", "feeDescription", "paymentStatus", "paidAmount", "hearingCentre",
    "isIntegrated", "isAdmin", "isEjp", "legalRepEmail", "legalRepGivenName", "legalRepCompanyPaperJ",
    "additionalInstructionsTribunalResponse", "additionalPaymentInfo", "additionalRequests",
    "additionalRequestsDescription", "isAdditionalAdjustmentsAllowed", "isAdditionalInstructionAllowed",
    "isHearingLoopNeeded", "isHearingRoomNeeded", "isInterpreterServicesNeeded", "isMultimediaAllowed",
    "isRemoteHearingAllowed", "isVulnerabilitiesAllowed", "vulnerabilitiesDecisionForDisplay",
    "multimediaDecisionForDisplay", "remoteHearingDecisionForDisplay", "singleSexCourt",
    "submissionOutOfTime", "recordedOutOfTimeDecision",
    
    # 2. NESTED STRUCTS (The ones previously missing or partial)
    F.col("listingLength.hours").alias("ll_h"),
    F.col("listingLength.minutes").alias("ll_m"),
    F.col("appellantAddress.AddressLine1").alias("app_addr1"),
    F.col("appellantAddress.PostCode").alias("app_pc"),
    F.col("legalRepAddressUK.AddressLine1").alias("rep_addr1"),
    F.col("legalRepAddressUK.PostCode").alias("rep_pc"),
    
    # NEW: Locations & Categories
    F.col("caseManagementLocation.baseLocation").alias("loc_base"),
    F.col("caseManagementLocation.region").alias("loc_region"),
    F.col("caseManagementCategory.value.label").alias("case_cat_label"),
    
    # NEW: Specific Responses
    "additionalTribunalResponse",
    "multimediaTribunalResponse",
    "vulnerabilitiesTribunalResponse",
    "remoteVideoCallTribunalResponse"
)

# --- Step B: Match Gold to Audit ---
# Ensure Gold has the EXACT same column list and aliases
gold_final = latest_json_df.select(
    "appealReferenceNumber",
    "appellantFullName", "appellantFamilyName", "appellantGivenNames",
    "appellantDateOfBirth", "appellantInUk", "appellantInDetention", "appellantStateless",
    "appealType", "appealTypeDescription", "appealOutOfCountry", "appealSubmissionDate",
    "feeAmountGbp", "feeDescription", "paymentStatus", "paidAmount", "hearingCentre",
    "isIntegrated", "isAdmin", "isEjp", "legalRepEmail", "legalRepGivenName", "legalRepCompanyPaperJ",
    "additionalInstructionsTribunalResponse", "additionalPaymentInfo", "additionalRequests",
    "additionalRequestsDescription", "isAdditionalAdjustmentsAllowed", "isAdditionalInstructionAllowed",
    "isHearingLoopNeeded", "isHearingRoomNeeded", "isInterpreterServicesNeeded", "isMultimediaAllowed",
    "isRemoteHearingAllowed", "isVulnerabilitiesAllowed", "vulnerabilitiesDecisionForDisplay",
    "multimediaDecisionForDisplay", "remoteHearingDecisionForDisplay", "singleSexCourt",
    "submissionOutOfTime", "recordedOutOfTimeDecision",
    F.col("listingLength.hours").alias("ll_h"),
    F.col("listingLength.minutes").alias("ll_m"),
    F.col("appellantAddress.AddressLine1").alias("app_addr1"),
    F.col("appellantAddress.PostCode").alias("app_pc"),
    F.col("legalRepAddressUK.AddressLine1").alias("rep_addr1"),
    F.col("legalRepAddressUK.PostCode").alias("rep_pc"),
    F.col("caseManagementLocation.baseLocation").alias("loc_base"),
    F.col("caseManagementLocation.region").alias("loc_region"),
    F.col("caseManagementCategory.value.label").alias("case_cat_label"),
    "additionalTribunalResponse",
    "multimediaTribunalResponse",
    "vulnerabilitiesTribunalResponse",
    "remoteVideoCallTribunalResponse"
)

# Apply trim to all string columns to avoid 'invisible' mismatches
for col_name, dtype in audit_final.dtypes:
    if dtype == "string":
        audit_final = audit_final.withColumn(col_name, F.trim(F.col(col_name)))
        gold_final = gold_final.withColumn(col_name, F.trim(F.col(col_name)))

# Convert all nulls to empty strings for a fairer comparison
audit_final = audit_final.na.fill("")
gold_final = gold_final.na.fill("")

In [0]:
audit_final.display()

audit_final.count()

In [0]:
audit_payload_df.display()

audit_payload_df.count()

In [0]:
gold_final.display()

gold_final.count()

In [0]:
latest_json_df.display()

latest_json_df.count()

In [0]:
# # Grab 10 rows
# seed_rows = gold_final.limit(10)

# # Extract reference numbers so we can remove them from the original set
# seed_refs = [row[0] for row in seed_rows.select("appealReferenceNumber").collect()]

# # Create seeded data
# seeded_rubbish_rows = seed_rows.withColumn("feeAmountGbp", F.lit("999999")) \
#                             .withColumn("appellantFullName", F.lit("Rubbish Data Batch")) \
#                             .withColumn("hearingCentre", F.lit("The Moon"))

# # Filter the original Gold to remove the 10 real versions of these cases
# gold_minus_victims = gold_final.filter(~F.col("appealReferenceNumber").isin(seed_refs))

# # Union the clean data with our 10 corrupted rows
# gold_with_seeds = gold_minus_victims.union(seeded_rubbish_rows)

# print(f"Targeting {len(seed_refs)} cases: {seed_refs}")
# print("10 Fake rows created and injected into 'gold_with_seeds'")

# # Overwrite for final test
# gold_final = gold_with_seeds

# gold_final.display()

In [0]:
# Final Test
mismatches = gold_final.subtract(audit_final)

if mismatches.count() == 0:
    print("Test pass: all fields match perfectly between gold and audit dataframe.")
else:
    print(f"Test fail: Found {mismatches.count()} rows with differences.") 
    print("The first dataframe shows the data in gold which did not match the audit table data.")
    print("Loading dataframe...")
    mismatches.display()

    # Extract appealReferenceNumbers
    mismatch_list = [row["appealReferenceNumber"] for row in mismatches.collect()]

    print(f"Searching for these {len(mismatch_list)} failing cases in the audit data:...")
    print("The next dataframe shows the audit table data for the caseNos which did not match in gold.")
    print("Loading dataframe...")
    audit_final.filter(col("appealReferenceNumber").isin(mismatch_list)).display()